# CS 3120/5120: Secure Distributed Computation
## Homework 3

In [ ]:
# Useful imports and utility functions
import pychor
import galois
p = 2**31-1
GF = galois.GF(p)

@pychor.local_function
def share(secret, n):
    shares = [GF.Random() for _ in range(n-1)]
    shares.append(GF(secret) - GF(shares).sum())
    return shares

## Question 1 (10 points)

Implement the ideal functionality for generating a multiplication triple with $n$ parties. Your solution should take a list of $n$ parties as input, and return the triple. The triple should be represented by 3 secret-shared values (for $a$, $b$, and $c$). Each secret-shared value should be represented by a dictionary mapping each party to their share.

In [ ]:
def functionality_gen_triple_n(parties):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
@pychor.local_function
def sum_shares(shares):
    return GF(shares).sum()

with pychor.LocalBackend():
    parties = [pychor.Party(f'p{i}') for i in range(1, 5)]
    a, b, c = functionality_gen_triple_n(parties)
    a_reconstructed = sum_shares(list(a.values()))
    b_reconstructed = sum_shares(list(b.values()))
    c_reconstructed = sum_shares(list(c.values()))

    assert a_reconstructed * b_reconstructed == c_reconstructed

## Question 2 (20 points)

Implement an $n$-party protocol for multiplying two secret-shared numbers using a multiplication triple. Both the input numbers and the multiplication triple will be represented by dictionaries mapping each participating party to their share, as in Question 1.

In [ ]:
def share_val(p, val, parties):
    n = len(parties)
    shares = dict(zip(parties, share(val, n).unlist(n)))
    for p2, s in shares.items():
        s.send(p, p2)
    return shares

def broadcast(p1, v):
    for p2 in parties:
        v.send(p1, p2)
    return v

def protocol_mult_n(parties, x, y, triple):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
with pychor.LocalBackend():
    parties = [pychor.Party(f'p{i}') for i in range(1, 5)]
    triple = functionality_gen_triple_n(parties)
    p0 = parties[0]
    x = share_val(p0, p0.constant(5), parties)
    y = share_val(p0, p0.constant(4), parties)
    z = protocol_mult_n(parties, x, y, triple)
    z_reconstructed = sum_shares([broadcast(p, z[p]) for p in parties])
    assert z_reconstructed.val == GF(20)

## Question 3 (20 points)

Implement a protocol to compute the *average* of a list of $m$ secret-shared numbers. The secret-shared numbers are shared among $n$ parties, and $n$ and $m$ may be different. Your protocol should reveal the final average to all of the $n$ participating parties. The value of $m$ is also public knowledge. The average is defined as:

$$\frac{1}{m} \sum_{i=1}^m x_i$$

Hint: consider revealing the sum of the $m$ numbers to all the parties, in order to avoid performing divison on secret-shared values.

In [ ]:
def protocol_average_n(parties, secret_shared_inputs):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
m = 30
n = 5
with pychor.LocalBackend():
    parties = [pychor.Party(f'p{i}') for i in range(1, n+1)]
    p0 = parties[0]
    values = list(range(1, m+1))
    secret_shared_inputs = [share_val(p0, p0.constant(v), parties) for v in values]
    result = protocol_average_n(parties, secret_shared_inputs)
    
    print('Result:', result)
    assert result.val == sum(values) / len(values)

## Question 4 (30 points)

Implement a protocol to compute the *variance* of a list of $m$ secret-shared numbers. The secret-shared numbers are shared among $n$ parties, and $n$ and $m$ may be different. Your protocol should reveal the variance to all of the $n$ participating parties. The value of $m$ is also public knowledge. The variance is defined as:

$$\frac{1}{m} \sum_{i=1}^m (x_i - \mu)^2$$

where $\mu$ is the average.

Hint: use `protocol_average_n` to compute and reveal the average, then compute the variance using `protocol_mult_n`. You'll need to truncate the average to an integer to encode it as a field element.

In [ ]:
def protocol_variance_n(parties, secret_shared_inputs):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
m = 300
n = 5
with pychor.LocalBackend():
    parties = [pychor.Party(f'p{i}') for i in range(1, n+1)]
    p0 = parties[0]
    values = list(range(1, m+1))
    secret_shared_inputs = [share_val(p0, p0.constant(v), parties) for v in values]
    result = protocol_variance_n(parties, secret_shared_inputs)
    
    average = int(sum(values) / len(values)) # truncated to an integer
    variance = sum([(v-average)**2 for v in values])/len(values)
    print('True variance:', variance)
    print('Protocol Result:', result)
    assert result.val == variance